# Amazon Reviews 2023 Quick Analysis

快速查看 `McAuley-Lab/Amazon-Reviews-2023` 的 `raw_review_Electronics` 小样本（5,000 条）。

In [ ]:
from pathlib import Path
import re
from collections import Counter

import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import display

plt.style.use('ggplot')
pd.set_option('display.max_colwidth', 180)

candidates = [
    Path('amazon_reviews_2023_electronics_sample_5k.parquet'),
    Path('nlp/data/amazon_reviews_2023_electronics_sample_5k.parquet'),
    Path('/scratch/rl182/nlp/data/amazon_reviews_2023_electronics_sample_5k.parquet'),
]

data_path = next((p for p in candidates if p.exists()), None)
if data_path is None:
    raise FileNotFoundError('Could not find amazon_reviews_2023_electronics_sample_5k.parquet')

df = pd.read_parquet(data_path)
df['timestamp'] = pd.to_datetime(df['timestamp'], unit='ms', errors='coerce')
df['review_chars'] = df['text'].fillna('').str.len()
df['review_words'] = df['text'].fillna('').str.split().str.len()
df['review_year'] = df['timestamp'].dt.year

print(f'Loaded: {data_path}')
print(f'Shape: {df.shape}')
display(df.head())

In [ ]:
summary = pd.DataFrame({
    'column': df.columns,
    'non_null': [df[c].notna().sum() for c in df.columns],
    'dtype': [str(df[c].dtype) for c in df.columns],
})
display(summary)

print('Rating distribution:')
display(df['rating'].value_counts(dropna=False).sort_index())

print('Verified purchase share:')
display(df['verified_purchase'].value_counts(dropna=False, normalize=True).rename('share'))

In [ ]:
sample_cols = ['rating', 'verified_purchase', 'helpful_vote', 'title', 'text']
sample_df = df[sample_cols].sample(8, random_state=42).reset_index(drop=True)
display(sample_df)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

df['rating'].value_counts().sort_index().plot(kind='bar', ax=axes[0], color='#4C78A8')
axes[0].set_title('Rating Distribution')
axes[0].set_xlabel('Rating')
axes[0].set_ylabel('Count')

df['review_words'].plot(kind='hist', bins=40, ax=axes[1], color='#F58518', edgecolor='black')
axes[1].set_title('Review Length Distribution (Words)')
axes[1].set_xlabel('Words per review')

plt.tight_layout()
plt.show()

In [ ]:
rating_stats = (
    df.groupby('rating')
    .agg(
        reviews=('rating', 'size'),
        avg_words=('review_words', 'mean'),
        avg_helpful_votes=('helpful_vote', 'mean'),
        verified_share=('verified_purchase', 'mean'),
    )
    .round(2)
)
display(rating_stats)

In [ ]:
stopwords = {
    'the', 'a', 'an', 'and', 'or', 'to', 'of', 'it', 'is', 'in', 'for', 'this', 'that',
    'i', 'was', 'with', 'my', 'but', 'on', 'are', 'not', 'have', 'had', 'be', 'as', 'so',
    'they', 'if', 'you', 'we', 'very', 'at', 'from', 'me', 'them', 'all', 'just', 'too',
    'these', 'its', 'would', 'one', 'will', 'about', 'can', 'when', 'up', 'out', 'has',
    'more', 'after', 'get', 'got', 'our', 'your', 'their', 'were', 'there', 'than'
}

tokens = re.findall(r"[a-z']+", ' '.join(df['text'].fillna('').str.lower().tolist()))
tokens = [t for t in tokens if len(t) >= 4 and t not in stopwords]
top_terms = pd.DataFrame(Counter(tokens).most_common(20), columns=['term', 'count'])
display(top_terms)

plt.figure(figsize=(10, 6))
plt.barh(top_terms['term'][::-1], top_terms['count'][::-1], color='#54A24B')
plt.title('Top Terms in Review Text')
plt.xlabel('Count')
plt.tight_layout()
plt.show()

In [ ]:
print('Quick takeaways:')
print('- Ratings skew can be inspected from the bar chart above.')
print('- Review length is highly variable, which is useful for retrieval and summarization tests.')
print('- Helpful votes and verified purchase can support evidence weighting later.')
print('- ASIN / parent ASIN fields make product-level aggregation possible.')